# Score a pipeline against the truth

Notebook 1 built a recording **forward** from its physics, and Notebook 2 ran a real
analysis pipeline on one. This notebook asks the question that makes a simulator
worth having: **how well did the pipeline recover the truth?** Because minisim
generated the recording, we hold the exact answer key - the true footprints,
calcium traces, spike counts, and motion - so we can grade a recovery instead of
guessing.

Grading sounds simple ("did it find the cells?") and is full of traps. A footprint
can overlap the right pixels yet smear its weight; a motion-corrected estimate can
sit a few pixels off the true frame through no fault of its own; the deconvolved `S`
a pipeline emits is **not** a spike train and must not be scored like one. This
notebook builds each recovery metric and, with it, the pitfall it is shaped to
avoid.

Two halves:

1. **Concepts and pitfalls** - each metric on controlled perturbations of the truth,
   so you *see* what it rewards and what it forgives.
2. **An end-to-end walkthrough** - mock an imperfect pipeline output, score it in one
   call, and read every line of the report.

> Unlike Notebook 1, this notebook is **static**: each demo sweeps a perturbation and
> plots the whole curve, so it reads correctly even without a live kernel. Every demo
> cell starts with a `# try:` knob - edit it and re-run to explore.

## Setup

We need the simulator (to make a recording with known truth) and the recovery
metrics. `minisim.testing.make_recording` is the one-call CI fixture from the
test-suite guide; the a-la-carte metrics (`hungarian_match`, `trace_pearson`,
`activity_similarity`, `shift_rmse`) and the one-call `score` come straight off the
top-level package.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import shift as ndshift

from minisim import (
    hungarian_match, trace_pearson, activity_similarity, shift_rmse,
    global_shift_from_trajectories,
)
from minisim.testing import make_recording, score, Estimate

%matplotlib inline
plt.rcParams["image.cmap"] = "magma"
plt.rcParams["figure.dpi"] = 100

# One small, fast, deterministic recording with ground truth. `detectable_subset()`
# is the fair target: cells too dim/deep to clear the noise floor are not a fair miss
# (Notebook 1's detectability stage), so we score recovery against the detectable cells.
rec = make_recording(n_cells=12, n_px=96, duration_s=4.0, seed=0)
gt = rec.ground_truth
det = gt.detectable_subset()
print(f"{gt.n_units} cells planted, {det.n_units} detectable; "
      f"movie {tuple(rec.observed.shape)}")


def shift_footprints(A, dy, dx):
    # Translate a footprint stack on its (height, width) axes, zero-filling the edge
    # (what a constant frame offset does to recovered footprints).
    return ndshift(np.asarray(A, float), (0, dy, dx), order=1, mode="constant", cval=0.0)


def footprint_similarity(est2d, true2d, *, metric="iou", energy_frac=0.9):
    # Single-footprint similarity under one metric, via the pairwise matcher.
    m = hungarian_match(est2d[None], true2d[None], metric=metric, energy_frac=energy_frac)
    return float(m.similarity_matrix[0, 0])


def busiest_cell(S):
    # Index of the most active detectable cell - the clearest one for trace/activity demos.
    return int(np.argmax(np.asarray(S).sum(axis=1)))

# Part 1 - the metrics, and the traps they avoid

Each section perturbs the truth in one controlled way and watches a metric respond.
The recipe is always the same: take the ground truth, break it deliberately, score
it, and read what the number does.

## 1.1 Matching cells - which estimate is which truth?

Every recovery score starts by deciding *which* recovered footprint corresponds to
*which* true cell. `hungarian_match` builds the pairwise spatial-overlap matrix
between the estimated and true footprints and solves the optimal one-to-one
assignment. From the match come the familiar detection scores:

- **recall** - fraction of true cells recovered (overlap above a threshold),
- **precision** - fraction of estimates that are real,
- **mean overlap** - average overlap over the matched pairs.

Match against `A_observed` - the optically degraded footprint, which is the most a
pipeline could recover - not the sharp, optics-free `A_planted`. With a flawless
estimate (the truth fed back in) the overlap matrix is diagonal and the scores are 1.

In [ ]:
def show_perfect_match():
    A = np.asarray(det.A_observed)
    m = hungarian_match(A, A)               # estimate == truth
    print(f"recall={m.recall():.2f}  precision={m.precision():.2f}  "
          f"mean overlap={m.mean_similarity:.2f}  ({len(m.pairing)} pairs)")
    fig, ax = plt.subplots(figsize=(4.6, 4))
    im = ax.imshow(m.similarity_matrix, vmin=0, vmax=1, cmap="viridis")
    ax.set(xlabel="true cell", ylabel="estimated cell",
           title="footprint overlap matrix (diagonal = correct pairing)")
    fig.colorbar(im, ax=ax, label="IoU")
    plt.show()


show_perfect_match()

## 1.2 Pitfall: pixel weights matter

A footprint is not a flat blob - its bright core carries most of the cell's
identity, and a recovered footprint that lights the right *pixels* but spreads its
*weight* wrong is a worse match than its silhouette suggests. The default overlap is
binary **IoU** (Jaccard of energy masks), which sees only which pixels are lit. Two
weighted similarities compare the intensity profile instead:

- **cosine** - the angle between the flattened footprints (scale-free),
- **weighted Jaccard** - `sum(min) / sum(max)` of the (sum-normalized) footprints.

Below we keep a real footprint's *support* fixed and flatten its weight profile
(raise it to a power `gamma`: `gamma=1` is the true profile, `gamma -> 0` is a flat
disk). Binary IoU never notices; the weighted metrics fall as the profile degrades -
exactly the discrimination you want.

In [ ]:
def show_pixel_weights():
    # try: pick a different detectable cell, or a different range of gammas.
    u = busiest_cell(det.S)
    true = np.asarray(det.A_observed[u])
    support = true > 0
    gammas = np.linspace(1.0, 0.0, 11)      # 1 = true profile, 0 = flat over the support
    series = {"iou (binary)": [], "cosine": [], "weighted_jaccard": []}
    for g in gammas:
        est = np.where(support, true ** g, 0.0)
        # energy_frac=1 keeps the whole support, so the binary masks are identical here.
        series["iou (binary)"].append(footprint_similarity(est, true, metric="iou", energy_frac=1.0))
        series["cosine"].append(footprint_similarity(est, true, metric="cosine"))
        series["weighted_jaccard"].append(footprint_similarity(est, true, metric="weighted_jaccard"))

    fig, (axf, axc) = plt.subplots(1, 2, figsize=(10, 3.8))
    axf.imshow(np.where(support, true ** 0.0, 0.0)); axf.set_title("flattened estimate (gamma=0)")
    axf.set_xticks([]); axf.set_yticks([])
    for name, ys in series.items():
        axc.plot(gammas, ys, marker="o", ms=3, label=name)
    axc.invert_xaxis()
    axc.set(xlabel="weight profile gamma (1=true  ->  0=flat)", ylabel="similarity to truth",
            title="binary IoU is blind to the weight profile", ylim=(0, 1.02))
    axc.legend(fontsize=8); axc.grid(alpha=0.3)
    plt.show()


show_pixel_weights()

## 1.3 Pitfall: a global shift is not a miss

After motion correction the recovered footprints live in the pipeline's *template*
frame, which can sit a few pixels off minisim's reference - and the offset differs
between pipelines (rigid vs non-rigid registration, different templates). Scored
naively, a uniform translation of every footprint looks like a total failure even
when the recovery is otherwise perfect.

`hungarian_match` absorbs it. Pass `shift="auto"` to find the global translation that
maximizes overlap (robust to a few missed or spurious cells, since the shared mass
dominates), or pass a known `(dy, dx)`. Below we slide a perfect estimate sideways
and score it both ways: raw recall collapses as the offset grows; `shift="auto"`
holds it flat.

In [ ]:
def show_global_shift():
    # try: widen the offset range, or set metric="cosine".
    A = np.asarray(det.A_observed)
    offsets = range(0, 9)
    raw, auto, found = [], [], []
    for k in offsets:
        A_off = shift_footprints(A, k, k)            # every footprint moved by (k, k) px
        raw.append(hungarian_match(A_off, A).recall())
        m = hungarian_match(A_off, A, shift="auto")
        auto.append(m.recall())
        found.append(m.shift)
    fig, ax = plt.subplots(figsize=(6, 3.8))
    ax.plot(list(offsets), raw, marker="o", label="no shift handling")
    ax.plot(list(offsets), auto, marker="s", label='shift="auto"')
    ax.set(xlabel="global offset applied (px, both axes)", ylabel="recall",
           title="a uniform offset is recovered, not penalized", ylim=(-0.02, 1.05))
    ax.legend(); ax.grid(alpha=0.3)
    plt.show()
    print("recovered shifts (dy, dx):", found[:6], "...")


show_global_shift()

When the pipeline reports its **motion trajectory**, the offset need not be searched
for at all: the constant difference between the estimated correction and the true
applied motion *is* the footprint offset. `global_shift_from_trajectories` reads it
straight off, and `score` (Part 2) uses it automatically when both trajectories are
present.

## 1.4 Recovering the calcium trace

For each matched pair, `trace_pearson` correlates the recovered calcium trace against
the true `C`. Pearson r is invariant to scale and offset, so it scores the *shape* of
the recovered dynamics (a pipeline's trace is in arbitrary fluorescence units). Here
we corrupt a true trace with increasing noise and watch the correlation fall.

In [ ]:
def show_trace_recovery():
    # try: change the noise levels, or the cell.
    u = busiest_cell(det.S)
    true = np.asarray(det.C[u]); rng = np.random.default_rng(1)
    sd = true.std()
    fig, (axt, axc) = plt.subplots(1, 2, figsize=(10, 3.6))
    t = np.arange(true.size)
    for frac in (0.0, 0.5, 1.5):
        est = true + rng.normal(0, frac * sd, true.size)
        r = trace_pearson(est[None], true[None], ((0, 0),))[0]
        axt.plot(t, est, lw=0.8, label=f"noise {frac:g}x  (r={r:.2f})")
    axt.plot(t, true, color="k", lw=1.4, label="true C")
    axt.set(xlabel="frame", ylabel="fluorescence (a.u.)", title="recovered trace vs truth")
    axt.legend(fontsize=7)
    fracs = np.linspace(0, 2, 21)
    rs = [trace_pearson((true + rng.normal(0, f * sd, true.size))[None], true[None], ((0, 0),))[0]
          for f in fracs]
    axc.plot(fracs, rs, marker="o", ms=3); axc.set(xlabel="noise (x trace std)",
             ylabel="trace correlation r", title="r degrades with noise", ylim=(0, 1.02))
    axc.grid(alpha=0.3)
    plt.show()


show_trace_recovery()

## 1.5 Pitfall: the deconvolved `S` is not a spike train

The array a pipeline calls `S` (CNMF/minian "spikes") is **not** a list of action
potentials. It is a non-negative estimate of neural **activity rate**, scaled by an
unknown factor that maps activity to calcium-kernel amplitude. Two consequences for
scoring:

1. **Do not binarize it.** Thresholding `S` into "spike / no spike" throws away the
   graded amplitude, which is most of the information, and makes the score hostage to
   an arbitrary threshold.
2. **Allow an unknown scale.** The estimate and the truth differ by a gain factor, so
   the metric must be invariant to it (and can usefully *report* it).

`activity_similarity` does both. Per matched pair it returns the (scale/offset-
invariant) Pearson **correlation**, the recovered non-negative **scale** (the unknown
gain, made explicit), and the **variance explained** by that scaled fit. The true
target is `gt.S` - the per-frame spike *counts* from the biology (those are real
spikes); what the pipeline emits to approximate it is the non-spike activity estimate.

In [ ]:
def show_activity_is_not_spikes():
    # The true per-frame spike counts (real, integer) vs a plausible deconvolved
    # estimate: a non-negative, graded activity rate at an arbitrary gain.
    u = busiest_cell(det.S)
    true = np.asarray(det.S[u]); t = np.arange(true.size)
    est = 0.4 * true                       # same shape, unknown smaller gain
    act = activity_similarity(est[None], true[None], ((0, 0),))
    fig, ax = plt.subplots(figsize=(8.5, 3.2))
    ax.stem(t, true, basefmt=" ", linefmt="0.7", markerfmt=" ", label="true spike counts (gt.S)")
    ax.plot(t, est, color="C3", lw=1.2, label="deconvolved activity estimate (gain 0.4)")
    ax.set(xlabel="frame", ylabel="activity", title=(
        f"corr={act.correlation[0]:.2f}  recovered scale={act.scale[0]:.2f}  "
        f"variance explained={act.variance_explained[0]:.2f}"))
    ax.legend(fontsize=8)
    plt.show()


show_activity_is_not_spikes()

### Scale invariance: the metric recovers the unknown gain

Take a true activity and define the estimate as `true / k` for a range of gains `k`.
A scale-aware metric should report **correlation ~ 1** and **variance explained ~ 1**
for every `k` (the shape is perfect), while the recovered **scale** tracks `k` - it
has measured the unknown gain.

In [ ]:
def show_scale_invariance():
    # try: change the gain range.
    u = busiest_cell(det.S)
    true = np.asarray(det.S[u], float)
    ks = np.linspace(0.25, 4.0, 16)
    corr, pve, scale = [], [], []
    for k in ks:
        est = true / k                      # estimate is the truth at gain 1/k
        a = activity_similarity(est[None], true[None], ((0, 0),))
        corr.append(a.correlation[0]); pve.append(a.variance_explained[0]); scale.append(a.scale[0])
    fig, (axl, axr) = plt.subplots(1, 2, figsize=(10, 3.6))
    axl.plot(ks, corr, marker="o", ms=3, label="correlation")
    axl.plot(ks, pve, marker="s", ms=3, label="variance explained")
    axl.set(xlabel="applied gain k", ylabel="score", title="invariant to scale", ylim=(0, 1.05))
    axl.legend(); axl.grid(alpha=0.3)
    axr.plot(ks, scale, marker="o", ms=3, color="C2")
    axr.plot(ks, ks, ls="--", color="0.6", label="y = k")
    axr.set(xlabel="applied gain k", ylabel="recovered scale", title="the metric recovers the gain")
    axr.legend(); axr.grid(alpha=0.3)
    plt.show()


show_scale_invariance()

### Why not binarize: amplitude errors a threshold cannot see

Two estimates fire on the **same frames**: one matches the true amplitudes, the other
flattens every event to height 1. A threshold-based spike precision/recall calls both
*perfect* (identical nonzero frames). `activity_similarity` does not: the flattened
estimate explains less of the true variance, because it gets the amplitudes wrong.

In [ ]:
def naive_spike_precision_recall(est, true, thr=0.0):
    # The OLD, binarized approach, for contrast only: a frame is a "spike" if it
    # exceeds thr. It is blind to amplitude.
    e = np.asarray(est) > thr; t = np.asarray(true) > thr
    tp = int((e & t).sum())
    prec = tp / int(e.sum()) if e.sum() else 0.0
    rec = tp / int(t.sum()) if t.sum() else 0.0
    return prec, rec


def show_no_binarize():
    u = busiest_cell(det.S)
    true = np.asarray(det.S[u], float)
    graded = true.copy()                    # right frames, right amplitudes
    flat = (true > 0).astype(float)         # right frames, wrong (flattened) amplitudes
    for name, est in [("graded (correct amplitudes)", graded), ("flattened (amplitudes lost)", flat)]:
        a = activity_similarity(est[None], true[None], ((0, 0),))
        p, r = naive_spike_precision_recall(est, true)
        print(f"{name:32s}  binarized P/R = {p:.2f}/{r:.2f}   "
              f"variance explained = {a.variance_explained[0]:.2f}")
    print("\nThe binarized score cannot tell the two apart; variance explained can.")


show_no_binarize()

## 1.6 Motion: score the tracking, not the origin

If the spec has brain motion, `gt.shifts` is the true per-frame `(dy, dx)`.
`shift_rmse` compares a pipeline's estimated trajectory against it, with two flags
that matter:

- `correction=True` negates the estimate - a motion-*correction* trajectory is the
  negation of the applied motion (it is what you would apply to undo it).
- `align=True` removes a constant per-axis offset before the RMSE. Each pipeline
  registers to its own template, so the absolute zero frame is arbitrary; aligning
  scores how well the *motion* was tracked rather than which frame was called zero.

Below, a trajectory that tracks the motion perfectly but sits at a constant offset
scores a large raw RMSE and ~0 once aligned.

In [ ]:
def show_motion():
    # try: change the constant offset, or add per-frame noise to `est`.
    recm = make_recording(n_cells=6, n_px=96, duration_s=4.0, motion=True, seed=1)
    true = np.asarray(recm.ground_truth.shifts)
    correction = -true                       # a perfect correction undoes the motion
    correction = correction + np.array([3.0, -2.0])   # but registered to a shifted template
    raw = shift_rmse(correction, true, correction=True)
    aligned = shift_rmse(correction, true, correction=True, align=True)
    offset = global_shift_from_trajectories(correction, true)
    print(f"raw RMSE = {raw:.2f} px    aligned RMSE = {aligned:.2f} px")
    print(f"constant offset read off the trajectories (-> footprint shift): {offset}")
    fig, ax = plt.subplots(figsize=(7, 3.2))
    t = np.arange(true.shape[0])
    ax.plot(t, true[:, 0], color="k", label="true dy")
    ax.plot(t, (-correction)[:, 0], color="C1", ls="--", label="estimated dy (negated)")
    ax.set(xlabel="frame", ylabel="shift dy (px)", title="same motion, constant origin offset")
    ax.legend(fontsize=8)
    plt.show()


show_motion()

# Part 2 - score a recovery end to end

Part 1 used one metric at a time on a clean perturbation. In practice you run a
pipeline, get back footprints / traces / activity / a motion trajectory, and want a
single honest scorecard. That is `minisim.testing.score`, which bundles every metric
above and applies the conventions you would otherwise have to remember (match against
`A_observed`, score recall over the detectable cells, absorb a global footprint
offset, score activity up to scale, align the motion origin).

## 2.1 Mock an imperfect pipeline output

To keep the notebook dependency-free we do not run a real pipeline here (Notebook 2
does). Instead we *fake* a realistic recovery by breaking the truth the way a pipeline
would: miss a dim cell, invent a spurious one, shift the footprints, add trace noise,
and emit the activity at an unknown gain. `Estimate` is the container `score` takes;
its fields accept either the terse CNMF symbols (`A`, `C`, `S`) or spelled-out aliases
(`footprints`, `traces`, `activity`).

In [ ]:
def mock_recovery(gt, *, n_miss=1, n_false=1, template_shift=(3, 2), trace_noise=0.3,
                  activity_gain=2.0, seed=2):
    # Build a plausible-but-imperfect Estimate from the detectable truth. If the
    # recording has motion, the recovered footprints sit at a constant `template_shift`
    # off minisim's reference (a different registration template) and the estimated
    # correction over-shoots the true motion by that same constant - so the offset is
    # real but recoverable from the trajectory.
    rng = np.random.default_rng(seed)
    det = gt.detectable_subset()
    A = np.asarray(det.A_observed); C = np.asarray(det.C); S = np.asarray(det.S)
    has_motion = det.shifts is not None
    keep = slice(n_miss, None)                       # "miss" the first n_miss cells
    A_keep, C_keep, S_keep = A[keep], C[keep], S[keep]
    fp_shift = template_shift if has_motion else (0, 0)
    A_est = shift_footprints(A_keep, *fp_shift)
    C_est = C_keep + rng.normal(0, trace_noise * C_keep.std(axis=1, keepdims=True), C_keep.shape)
    S_est = S_keep / activity_gain                   # activity at an unknown gain
    shifts_est = (-np.asarray(det.shifts) + np.array(template_shift)) if has_motion else None
    if n_false:                                      # invent spurious footprints in empty corners
        h, w = A.shape[1:]; false = np.zeros((n_false, h, w))
        for i in range(n_false):
            cy, cx = rng.integers(12, h - 12), rng.integers(12, w - 12)
            yy, xx = np.ogrid[:h, :w]
            false[i] = np.exp(-((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * 4.0 ** 2))
        A_est = np.concatenate([A_est, false])
        C_est = np.concatenate([C_est, rng.normal(0, 1, (n_false, C.shape[1]))])
        S_est = np.concatenate([S_est, np.abs(rng.normal(0, 0.1, (n_false, S.shape[1])))])
    return Estimate(A=A_est, C=C_est, activity=S_est, shifts=shifts_est)


# A recording WITH brain motion, so the recovery exercises the motion trajectory and
# the global footprint offset it implies (the SpecWarning notes the motion is large
# relative to this small teaching FOV; that is fine here).
rec2 = make_recording(n_cells=12, n_px=96, duration_s=4.0, motion=True, seed=0)
gt2 = rec2.ground_truth
est = mock_recovery(gt2)
print(f"estimate: {np.asarray(est.A).shape[0]} footprints "
      f"(truth has {gt2.detectable_subset().n_units} detectable, with motion)")

## 2.2 Score it in one call

`score(estimate, ground_truth)` returns a `Report`. Its `summary()` is a compact,
test-friendly digest; every field is also a plain attribute.

In [ ]:
report = score(est, gt2)
print(report.summary())

Reading the report field by field:

- **recall / precision / f1** - detection quality. The missed cell costs recall; the
  spurious footprint costs precision.
- **mean_overlap** - average footprint overlap over the matched pairs (under
  `match_metric`, IoU by default).
- **footprint_shift** - the global `(dy, dx)` `score` absorbed before matching, read
  off the motion trajectory, so the template offset we baked in did not masquerade as
  missed cells.
- **trace_corr** - median trace correlation over matched pairs.
- **activity_corr / activity_variance_explained / activity_scale** - the deconvolved
  activity scored up to scale; `activity_scale` is the recovered gain (~2, the gain we
  applied).
- **shift_rmse** - motion error in pixels (origin-aligned); small here, since the mock
  correction tracks the true motion up to the constant template offset.

In [ ]:
for f in ("recall", "precision", "f1", "mean_overlap", "footprint_shift",
          "trace_corr", "activity_corr", "activity_variance_explained",
          "activity_scale", "shift_rmse"):
    print(f"{f:30s} {getattr(report, f)}")

## 2.3 The honest denominator

`recall = 1.0` is only meaningful if you know what it is over. By default `score`
counts recall against the **detectable** cells, and the report always carries the
three counts so a high recall over a shrunken denominator can never be mistaken for
"recovered everything":

- `n_requested` - cells planted,
- `n_detectable` - cells above the detection floor,
- `n_true` - the denominator recall actually used.

Pass `restrict_to_detectable=False` to grade against every planted cell instead (the
undetectable ones then count as misses).

In [ ]:
print(f"planted={report.n_requested}  detectable={report.n_detectable}  "
      f"recall denominator={report.n_true}")
full = score(est, gt2, restrict_to_detectable=False)
print(f"recall over detectable={report.recall:.2f}   over all planted={full.recall:.2f} "
      f"(denominator {full.n_true})")

## 2.4 Adapt it to your pipeline

For a real run, replace the mock with your pipeline's output and pass it straight to
`score`. The footprints are the only required field; leave traces / activity / shifts
out and those scores come back as `nan` / `None`.

```python
from minisim import simulate
from minisim.testing import Estimate, score

rec = simulate(spec)
A_est, C_est, S_est, shifts_est = run_my_pipeline(rec.observed)

report = score(
    Estimate(A=A_est, C=C_est, activity=S_est, shifts=shifts_est),
    rec.ground_truth,
    match_metric="cosine",     # weigh pixel weights, not just lit pixels (optional)
)
assert report.recall > 0.8, report.summary()
```

When both the estimated and true motion trajectories are present, `score` reads the
global footprint offset straight off them (Section 1.3); otherwise it estimates the
offset by overlap. The {doc}`benchmarking guide <../../../docs/howto/benchmark>` has
the same recipe outside a notebook, and the test-suite guide wires it into `pytest`.

## 2.5 Scaling up: a metric across a physical axis

A single scorecard answers "how did this run do?". The deeper questions are physical:
how does recall fall with **depth**, with **density**, with **NA**? Drive this same
`score` call from a parameter sweep (one recording per point on the axis) and collect
the reports into a table - recall vs depth is the canonical curve, and it is where the
provisional detection threshold gets calibrated against a real pipeline.

That sweep is the natural sequel to this notebook: the metrics here are the `y`-axis
of every such study.

In [ ]:
def show_recall_vs_depth():
    # try: change the depths, or n_cells. One fresh recording + mock recovery per depth.
    depths = [30, 60, 90, 120, 150]
    recalls = []
    for d in depths:
        r = make_recording(n_cells=12, n_px=96, duration_s=2.0, depth_um=float(d), seed=0)
        rep = score(mock_recovery(r.ground_truth), r.ground_truth)
        recalls.append(rep.recall)
    fig, ax = plt.subplots(figsize=(6, 3.6))
    ax.plot(depths, recalls, marker="o")
    ax.set(xlabel="cell depth (um)", ylabel="recall (over detectable)",
           title="recovery vs depth - the canonical sweep", ylim=(-0.02, 1.05))
    ax.grid(alpha=0.3)
    plt.show()


show_recall_vs_depth()

## Recap

- **Matching** pairs estimates to truth by spatial overlap; recall / precision / f1
  follow. Use **weighted** overlap (`cosine`, `weighted_jaccard`) when pixel weights
  matter, and **`shift`** to absorb a global offset after motion correction.
- **Traces** are scored by scale-invariant correlation.
- **Activity** (`S`) is a scaled rate, not spikes: score it without binarizing and up
  to an unknown gain, which `activity_similarity` reports.
- **Motion** is scored with `shift_rmse`, aligning away the arbitrary origin.
- **`score`** bundles all of this into one honest `Report`, with the recall
  denominator always in view.

These metrics are the answer key Notebook 1's forward simulation exists to provide,
and the `y`-axis of every benchmarking study built on minisim.